The primary objective of this pipeline is to curate a standardized Star Schema optimized for high-performance Business Intelligence (BI) and advanced analytics. It centralizes time-series market metrics and provides rich geographic context.

Key Architecture Components
**Target Catalog:** data_gold

**Target Schema:** data_stg (Staging)

SCD Logic: Type 1 (Overwrite/Current State)

Storage Format: Delta Lake

In [0]:
import warnings
import sys
from pyspark.sql import functions as F
from pyspark.sql import Window
from pyspark.sql.types import StringType, IntegerType
from pyspark.sql.utils import AnalysisException
from state_mapping import STATE_MAP, cities_dim, county_dim, cols_to_drop, common_metrics, fact_configs

# --- 1. TARGET CONFIGURATION ---
# Gold represents the 'Consumption' layer where tables are structured for end-users.
TARGET_CATALOG = "data_gold"
TARGET_SCHEMA  = "data_stg"
BASE_PATH      = f"{TARGET_CATALOG}.{TARGET_SCHEMA}"

try:
    # Ensure the Gold schema exists; this is the final destination for BI reports.
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {BASE_PATH}")
except Exception as e:
    print(f"Error creating schema {BASE_PATH}: {str(e)}")
    sys.exit(1)

# --- 2. FACT CONSOLIDATION (Unified Market Data) ---
# This section takes multiple Zillow grains (Zip, City, etc.) and stacks them into one Fact table.
fact_dfs = []
try:
    if not fact_configs:
        raise ValueError("fact_configs dictionary is empty; no source tables defined.")
    
    for grain, table_path in fact_configs.items():
        try:
            # Load each silver table and inject the 'grain' (e.g., 'zip', 'city') 
            # as a column to differentiate data sources in the unified view.
            df = spark.read.table(table_path) \
                .withColumn("region_type", F.lit(grain)) \
                .withColumn("regionname", F.col("regionname").cast(StringType()))
            
            # Select only the metrics common across all grains to ensure a clean union.
            target_cols = [c for c in common_metrics if c in df.columns] + ["region_type"]
            fact_dfs.append(df.select(target_cols))
        except AnalysisException:
            print(f"Warning: Table {table_path} not found. Skipping grain: {grain}")
            continue

    if not fact_dfs:
        raise Exception("No fact dataframes were successfully loaded.")

    # UnionByName merges DataFrames while aligning columns, allowing for missing columns if necessary.
    unified_fact_base = fact_dfs[0]
    for next_df in fact_dfs[1:]:
        unified_fact_base = unified_fact_base.unionByName(next_df, allowMissingColumns=True)

    # Remove potential duplicates across grains to ensure 'One Version of the Truth'.
    unified_fact_base = unified_fact_base.distinct()

except Exception as e:
    print(f"Failed during fact consolidation: {str(e)}")
    sys.exit(1)

# --- 3. BUILD DIMENSION BASES (Geography & Dictionary) ---
try:
    # Load raw dimension sources from Silver.
    df_cities = spark.read.table(cities_dim)
    df_county = spark.read.table(county_dim)
    
    if not STATE_MAP:
        raise ValueError("STATE_MAP is missing; cannot map State abbreviations to full names.")
        
    # Build a Spark SQL Map from a Python dictionary to perform high-speed name lookups.
    mapping_expr = F.create_map([F.lit(x) for x in sum(STATE_MAP.items(), ())])

    # Standardize city names and resolve full state names (e.g., 'NY' -> 'New York').
    df_cities_refined = df_cities.drop(*cols_to_drop).withColumn("statename_mapped", mapping_expr[F.col("state")])

    # Join City and County data to create a 'Geography Master' dimension.
    # This join ensures that the Geography dim is consistent across different Zillow grains.
    dim_geography_base = df_cities_refined.join(
        df_county.drop(*cols_to_drop),
        (df_cities_refined.county == df_county.countyname) & 
        (df_cities_refined.statename_mapped == df_county.statename),
        "inner"
    ).drop("statename_mapped", "countyname", "statename").distinct()

    # Load the Data Dictionary as a 'Metric Glossary' dimension for end-user self-service.
    dim_dict_base = spark.read.table("data_silver.silver.dim_datadictionary") \
        .select(F.col("variable").alias("metric_name"), "definition").distinct()

except Exception as e:
    print(f"Error building dimension bases: {str(e)}")
    sys.exit(1)

# --- 4. REFINED METADATA & SURROGATE KEYS ---

def finalize_gold_dimension(df, source_name):
    """
    Adds surrogate keys (row_id) and audit metadata to dimensions.
    """
    try:
        warnings.filterwarnings("ignore", category=UserWarning, message=".*Window.*")
        # Generate a unique integer ID for the dimension using Windowing.
        window_spec = Window.orderBy(F.monotonically_increasing_id())
        df = df.withColumn("row_id", F.row_number().over(window_spec).cast(IntegerType()))
        
        # Ensure row_id is the first column in the table (BI Best Practice).
        cols = ["row_id"] + [c for c in df.columns if c != "row_id"]
        return df.select(cols) \
                 .withColumn("load_dt", F.current_timestamp()) \
                 .withColumn("source", F.lit(source_name))
    except Exception as e:
        print(f"Error in finalize_gold_dimension for {source_name}: {str(e)}")
        raise

def finalize_gold_fact(df, source_name):
    """
    Organizes columns and adds audit metadata to the Fact table.
    """
    try:
        # Move join-keys to the front of the DataFrame for better readability.
        priority_cols = ["regionname", "date", "region_type"]
        other_cols = [c for c in df.columns if c not in priority_cols]
        return df.select(priority_cols + other_cols) \
                 .withColumn("load_dt", F.current_timestamp()) \
                 .withColumn("source", F.lit(source_name))
    except Exception as e:
        print(f"Error in finalize_gold_fact for {source_name}: {str(e)}")
        raise

# Process all tables through the metadata layer.
dim_geography_gold = finalize_gold_dimension(dim_geography_base, "geo_crosswalk_pipeline")
dim_dict_gold      = finalize_gold_dimension(dim_dict_base, "metric_dictionary_pipeline")
fact_market_metrics_gold = finalize_gold_fact(unified_fact_base, "consolidated_fact_pipeline")

# --- 5. STORAGE & OPTIMIZATION (Delta Lake) ---

def save_gold_table(df, table_name, partitions=None):
    """
    Saves tables as Delta with optional partitioning.
    """
    try:
        writer = df.write.format("delta").mode("overwrite")
        if partitions:
            writer = writer.partitionBy(*partitions)
        writer.saveAsTable(table_name)
        print(f"Successfully saved {table_name}")
    except Exception as e:
        print(f"Failed to save table {table_name}: {str(e)}")
        raise

# Partitioning strategies:
# Dimensions: Partitioned by Geo-hierarchy to speed up location-based filtering.
# Facts: Partitioned by grain (region_type) and date to speed up trend reports.
save_gold_table(dim_geography_gold, f"{BASE_PATH}.dim_geography", ["state", "county", "city"])
save_gold_table(dim_dict_gold, f"{BASE_PATH}.dim_metric_dictionary")
save_gold_table(fact_market_metrics_gold, f"{BASE_PATH}.fact_market_metrics", ["region_type", "date"])

# OPTIMIZATION (Z-ORDER):
# Physically reorganize data for columns frequently used in WHERE clauses to maximize 'Data Skipping'.
for table, z_col in [("dim_geography", "row_id"), 
                     ("dim_metric_dictionary", "row_id"), 
                     ("fact_market_metrics", "regionname")]:
    try:
        spark.sql(f"OPTIMIZE {BASE_PATH}.{table} ZORDER BY ({z_col})")
    except Exception as e:
        print(f"Optimization failed for {table}: {str(e)}")

# --- 6. ENTERPRISE DATA GOVERNANCE ---

def apply_metadata(table_name, table_desc, col_comments):
    """
    Applies DDL comments to the table and columns for documentation in Unity Catalog.
    """
    try:
        spark.sql(f"COMMENT ON TABLE {table_name} IS '{table_desc}'")
        for col, comment in col_comments.items():
            spark.sql(f"ALTER TABLE {table_name} ALTER COLUMN `{col}` COMMENT '{comment}'")
    except Exception as e:
        print(f"Metadata application failed for {table_name}: {str(e)}")

apply_metadata(f"{BASE_PATH}.dim_geography", "Master Geography crosswalk for all grains.", {"row_id": "Surrogate primary key."})
apply_metadata(f"{BASE_PATH}.dim_metric_dictionary", "Business glossary for housing metrics.", {"metric_name": "Physical column name."})
apply_metadata(f"{BASE_PATH}.fact_market_metrics", "Consolidated housing metrics (Zip, City, Metro).", {"regionname": "Key to join with Geography Dim."})

print(f"Gold Layer built successfully in {BASE_PATH}.")

In [0]:
%sql
ALTER TABLE data_gold.data_stg.dim_geography SET TBLPROPERTIES (delta.enableChangeDataFeed = true);
ALTER TABLE data_gold.data_stg.dim_geography SET TBLPROPERTIES (delta.enableChangeDataFeed = true);
ALTER TABLE data_gold.data_stg.dim_metric_dictionary SET TBLPROPERTIES (delta.enableChangeDataFeed = true);

-- Add this as a notebook step 2 after moving data to data_stg schema in data_gold and before running the pipeline


In [0]:
%sql
-- Adding this as step 3 before running the DLT pipeline
-- Also we'll be adding some performance optimization on staging tables (specially for dim_geography since it does string to string comparison along with SCD-2 updates, so it takes the most time for refresh part in DLT full refresh): Done
-- 1. Enabling CDF
-- Enable CDF on Geography Staging
ALTER TABLE data_gold.data_stg.dim_geography 
SET TBLPROPERTIES (delta.enableChangeDataFeed = true);

-- Enable CDF on Dictionary Staging
ALTER TABLE data_gold.data_stg.dim_metric_dictionary 
SET TBLPROPERTIES (delta.enableChangeDataFeed = true);

-- Optimize the layout of the staging table
OPTIMIZE data_gold.data_stg.dim_geography 
ZORDER BY (unique_city_id);


For 1st version of without partitioning and zorder of tables (same code used for table creation but without partitioning of any table), runtime of queries on different tables:
- data_gold.data_stg.fact_market_metrics = 1.90 secs
- data_gold.data_stg.dim_geography = 1.92 secs
- data_gold.data_stg.dim_metric_dictionary = 1.25 secs

Now we're focusing on running the queries on gold staging tables which are manually partitioned based on partitionining condition for dim geo and 1 fact table created

In [0]:
#%sql
#select * from data_gold.data_stg.dim_geography -- where county = 'Dutchess'
#-- 1.71 sec

In [0]:
#%sql
    
#select * from data_gold.data_stg.fact_market_metrics -- total data count: 10584674
#where region_type = 'city'
#and date = '2008-08-31'

#-- 1.75

In [0]:
#%sql
#select * from data_gold.data_stg.dim_metric_dictionary where metric_name = 'DaysOnZillow'
#-- 1.26


After partitioning runtime for same set of filter:
- data_gold.gold.fact_market_metrics = 1.75 secs ( partitioned on columns region_type and date )
- data_gold.gold.dim_geography = 1.71 secs ( partitioned on state, county and city since it is the hierachy to reach an address by conventional hierarchy )
- data_gold.gold.dim_metric_dictionary = 1.26 secs ( not partitioned and only 30 distinct rows)


**NOTE: Based on the dataset, not much of performance changes have been observed**
